# PHY224 — Notebook 02: Native-pixel phase retrieval (ASM and discretized Kirchhoff–Fresnel)

Inputs: original detector ZIPs and geometry saved by **Notebook 01**. Recovered physical method from `batch2_native_combinatorial.py`, `native_asm_200iter_batch2.py`, and `calibrated_day2.py` (verbatim source scripts in ZIP). The core alternating-plane procedure is:
1. RGB-baseline subtraction → only the **red** channel → intensity proxy → square-root amplitude.
2. Native-pixel affine registration; centre zero-pad to 2048×3072 for FFT **without downsampling the 2592×1944 sensor**.
3. Iterate z1↔z2 amplitude replacement, propagate backwards to object, clip real transmission 0–1, multiply by incident Gaussian, propagate forwards.
4. Repeat 200 iterations; threshold amplitude transmission **T > 0.50**; save reconstruction, binary mask, residuals, optional IoU against a *separate photographic proxy*.

This notebook contains a **new notebook adapter** for the recovered algorithms, because the original Batch 2 script used a preexisting near-camera reference mask and the earlier three-hole script downsampled fourfold and used RGB magnitude. The original scripts are included unchanged in `recovered_original_scripts/`. This notebook does not claim to reproduce published final numbers without executing the original archived runs. The near-image proxy is never used as a target inside the inversion.

**Physical caveat:** The camera BMPs are pseudocolour exports, not certified linear irradiance measurements. The sampled Kirchhoff kernel can be spatially aliased and should not be treated as a converged exact diffraction solution. Nominal distances ignore unmeasured sensor recess.

In [ ]:
from pathlib import Path
import itertools, csv, json, time, zipfile, math
import cv2, numpy as np
from scipy.fft import fft2,ifft2,fftfreq,ifftshift
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

SEARCH_DIRS=[Path.cwd(),Path('/mnt/data'),Path('/content')]
def find_input(name):
    for d in SEARCH_DIRS:
        p=d/name
        if p.is_file():return p
    raise FileNotFoundError(f'Missing {name}; upload beside notebook.')
BASE=Path.cwd()/'PHY224_outputs'
CAL_DIR=BASE/'calibration'; OUT=BASE/'phase_retrieval';OUT.mkdir(parents=True,exist_ok=True)
H,W=1944,2592
PH,PW=2048,3072
PITCH=2.775e-6
LAMBDA=632.8e-9
ITERATIONS=200
THRESHOLD=0.5
sl=(slice((PH-H)//2,(PH-H)//2+H),slice((PW-W)//2,(PW-W)//2+W))

# Configure the run here. 2-plane reference pairs (all combinations optional).
DATASET='seven'      # 'seven' or 'three'
METHODS=('ASM','Kirchhoff-discrete')
RUN_ALL_PAIRS=False   # True: all 3 or all 15 pairs, potentially expensive at native size
SELECTED_PAIR=('z25','z68')  # seven: z25,z31,z68; three: z22,z34,z59,z107,z118,z245
# No random-aperture z-label mapping has been independently verified; do not invent one.

## A. Read original full-resolution CMOS frames and apply RGB baseline before red extraction

In [ ]:
SEVEN_FILES={
 'near':'batch_2_binary mask.bmp', 'gauss':'batch_2_gaussianmask.bmp',
 'z25':'battch2_2.5cms.bmp', 'z31':'Batch2_3.1cms.bmp', 'z68':'batch2_6.8cms.bmp'}
SEVEN_Z={'z25':.025,'z31':.031,'z68':.068}
THREE_FILES={
 'near':'DAY_2_WITH MONITOR BINARY CAMERA STABLE.bmp',
 'gauss1':'DAY_2_GAUSSIAN_WITH MONITOR_BINARY MASK CAMRA STABLE_1.bmp',
 'gauss2':'DAY_2_GAUSSIAN_WITH MONITOR_BINARY MASK CAMRA STABLE_2.bmp',
 'blank':'day_2_baseline_forall z more than 10.7cms.bmp',
 'z22':'DAY_2_2.2cms With monitor.bmp', 'z34':'DAY_2_3.4cms withnitor.bmp',
 'z59':'DAY_2_5.9cms.bmp', 'z107':'DAY_2_10.7cms.bmp',
 'z118':'DAY_2_11.8cms.bmp', 'z245':'DAY_2_24.5cms.bmp'}
THREE_Z={'z22':.022,'z34':.034,'z59':.059,'z107':.107,'z118':.118,'z245':.245}
if DATASET=='seven':
    archive=find_input('barch_2_0.zip'); files=SEVEN_FILES; Z=SEVEN_Z
    # Original Batch 2 red-background correction: ~1 digital count in each channel.
    baseline=np.ones((1,1,3),np.float32)
elif DATASET=='three':
    archive=find_input('DAY_2_2.2cms With monitor.zip');files=THREE_FILES;Z=THREE_Z
else:raise ValueError('Configure DATASET to seven or three; random distances not verified')

def read_bgr_zip(z,name):
    a=cv2.imdecode(np.frombuffer(z.read(name),np.uint8),cv2.IMREAD_COLOR)
    if a is None or a.shape[:2]!=(H,W):raise ValueError(f'Expected {W}x{H}: {name}')
    return a.astype(np.float32)

def subtract_then_red(img,baseline):
    # True per-channel subtraction; only red (index 2) enters reconstruction.
    return np.maximum(img-baseline,0.0)[:,:,2].astype(np.float32)

with zipfile.ZipFile(archive) as z:
    if DATASET=='three':baseline=read_bgr_zip(z,files['blank'])
    near= subtract_then_red(read_bgr_zip(z,files['near']),baseline)
    if DATASET=='seven':
        gauss=subtract_then_red(read_bgr_zip(z,files['gauss']),baseline)
        # Same fitted translation retained from the earlier Batch2 implementation.
        M=np.array([[1,0,600],[0,1,-260]],np.float32)
        beam=gaussian_filter(cv2.warpAffine(gauss,M,(W,H)),40).astype(np.float32)
    else:
        bp=CAL_DIR/'day2_red_gaussian_beam.npz'
        if not bp.is_file():raise FileNotFoundError('Run Notebook 01 to generate '+str(bp))
        beam=np.load(bp)['beam'].astype(np.float32)
    print('Loaded near and Gaussian; native dimensions',near.shape)

def get_reg(name):
    if DATASET=='seven':
        p=CAL_DIR/f'native_registration_{name}.npz'
        if not p.is_file():raise FileNotFoundError(f'Run Notebook 01 to generate {p}')
        a=np.load(p)['affine'].astype(np.float32)
        with zipfile.ZipFile(archive) as z:
            image=subtract_then_red(read_bgr_zip(z,files[name]),baseline)
        im=cv2.warpAffine(image,a,(W,H),flags=cv2.INTER_LINEAR)
        v=cv2.warpAffine(np.ones((H,W),np.uint8),a,(W,H),flags=cv2.INTER_NEAREST)>0
        return im,v
    p=CAL_DIR/f'day2_registration_{name}.npz'
    if not p.is_file():raise FileNotFoundError(f'Run Notebook 01 to generate {p}')
    with np.load(p) as d:return d['red'].astype(np.float32),d['valid'].astype(bool)

## B. Construct Gaussian incident amplitude, independent aperture support, and optional near-camera reference

A bounding rectangle around observed diffraction landmarks constrains object support. The near-image mask is used **only after reconstruction for overlap scoring**, never as input to iterative updates.

In [ ]:
def make_padded(native, dtype=np.float32):
    out=np.zeros((PH,PW),dtype);out[sl]=native;return out

def reference_from_near(red, n_openings):
    sm=cv2.GaussianBlur(red,(0,0),7)
    binary=(sm>max(25.,float(sm.max())*.22)).astype(np.uint8)
    n,lab,s,c=cv2.connectedComponentsWithStats(binary,8)
    comps=sorted(((int(s[i,cv2.CC_STAT_AREA]),i)
                 for i in range(1,n) if s[i,cv2.CC_STAT_AREA]>300),reverse=True)
    return np.isin(lab,[idx for _,idx in comps[:n_openings]])

ref_mask=reference_from_near(near,7 if DATASET=='seven' else 3)
B=np.zeros((PH,PW),np.float32)
B[sl]=np.clip(np.sqrt(np.maximum(beam,0)/255.),.05,2.)
# Derive object support from labelled source-camera landmarks in the registration file.
if DATASET=='seven':
    with np.load(CAL_DIR/'native_registration_z25.npz') as d:
        pts=d['registered_points_src'].astype(np.float32)
    margin=100
else:
    with np.load(CAL_DIR/'day2_registration_z22.npz') as d:
        pts=d['reference_centers'].astype(np.float32)
    margin=180
xlo=max(0,int(pts[:,0].min()-margin));xhi=min(W,int(pts[:,0].max()+margin))
ylo=max(0,int(pts[:,1].min()-margin));yhi=min(H,int(pts[:,1].max()+margin))
support=np.zeros((PH,PW),bool)
support[(PH-H)//2+ylo:(PH-H)//2+yhi,(PW-W)//2+xlo:(PW-W)//2+xhi]=True
print('Support native bbox',xlo,xhi,ylo,yhi,'reference open px',int(ref_mask.sum()))
# When selecting a session, the 2-plane FFT grid retains every native sensor pixel.

## C. ASM and sampled Kirchhoff–Fresnel propagators

**ASM:** FFT → angular-spectrum transfer → inverse FFT. **Kirchhoff:** FFT-based convolution with a discretely sampled scalar obliquity-weighted kernel, as in the recovered source. The sampled kernel may alias; this is a numerical caveat, not a claim that Kirchhoff theory is incorrect.

In [ ]:
fx=fftfreq(PW,PITCH)
fy=fftfreq(PH,PITCH)
k=2*np.pi/LAMBDA
kz=np.sqrt(k*k-(2*np.pi*fx[None,:])**2-(2*np.pi*fy[:,None])**2+0j).astype(np.complex64)
x=(np.arange(PW)-PW//2)*PITCH
y=(np.arange(PH)-PH//2)*PITCH
TRANSFER={}
def kernel(method,dist):
    d=round(float(abs(dist)),8)
    key=(method,d)
    if key not in TRANSFER:
        if method=='ASM':
            Hf=np.exp(1j*d*kz).astype(np.complex64)
        elif method=='Kirchhoff-discrete':
            rr=np.sqrt(d*d+y[:,None]**2+x[None,:]**2)
            ker=np.exp(1j*k*rr)*(1+d/rr)/(2j*LAMBDA*rr)*(PITCH**2)
            Hf=fft2(ifftshift(ker),workers=2).astype(np.complex64)
        else:raise ValueError(method)
        TRANSFER[key]=Hf
    return TRANSFER[key]

def propagate(E,dist,method):
    if abs(float(dist))<1e-12:return E.copy()
    Hf=kernel(method,dist)
    if dist<0:Hf=Hf.conj()
    return ifft2(fft2(E,workers=2)*Hf,workers=2).astype(np.complex64,copy=False)

## D. Recovered alternating-amplitude algorithm — 200 cycles and binary T > 0.50

In [ ]:
def reconstruct(pair, method='ASM', iterations=ITERATIONS, plot=True):
    z1,z2=[Z[n] for n in pair]
    if not z1<z2:raise ValueError('Select the two planes in increasing z order.')
    raw1,v1=get_reg(pair[0]);raw2,v2=get_reg(pair[1])
    A1=make_padded(np.sqrt(np.maximum(raw1,0)/255.),np.float32)
    A2=make_padded(np.sqrt(np.maximum(raw2,0)/255.),np.float32)
    V1=make_padded(v1,bool);V2=make_padded(v2,bool)
    E=np.zeros((PH,PW),np.complex64)
    E[V1]=A1[V1].astype(np.complex64)
    T=np.zeros((PH,PW),np.float32)
    curve=[];tstart=time.monotonic()
    for it in range(iterations):
        E=propagate(E,z2-z1,method)
        E[V2]=A2[V2]*np.exp(1j*np.angle(E[V2]))
        E=propagate(E,z1-z2,method)
        E[V1]=A1[V1]*np.exp(1j*np.angle(E[V1]))
        E0=propagate(E,-z1,method)
        T[:]=0
        T[support]=np.clip(np.real(E0[support]/np.maximum(B[support],.12)),0,1)
        E=propagate(B*T,z1,method)
        if (it+1)%20==0 or it==0:
            pred=T[sl]>THRESHOLD
            u=int(np.count_nonzero(pred|ref_mask)); inter=int(np.count_nonzero(pred&ref_mask))
            curve.append({'iteration':it+1,'proxy_iou':inter/u if u else float('nan')})
            print(pair,method,'iteration',it+1,'photographic proxy IoU',curve[-1]['proxy_iou'],flush=True)
    recovered=(T[sl]>THRESHOLD)
    intersection=int(np.count_nonzero(recovered&ref_mask))
    union=int(np.count_nonzero(recovered|ref_mask))
    result={'dataset':DATASET,'z1':pair[0],'z2':pair[1],
            'z1_cm':100*z1,'z2_cm':100*z2,'method':method,
            'iterations':iterations,'threshold':THRESHOLD,
            'photographic_proxy_IoU':intersection/union if union else float('nan'),
            'intersection':intersection,'union':union,
            'native_width':W,'native_height':H,'pitch_um':2.775,
            'fft_width':PW,'fft_height':PH,
            'reference':'near-camera photographic brightness-component proxy; not verified microscope ground truth',
            'kernel_sampling_warning':method=='Kirchhoff-discrete',
            'elapsed_seconds':time.monotonic()-tstart}
    folder=OUT/DATASET;folder.mkdir(parents=True,exist_ok=True)
    stem=method.replace('-','_')+'_'+pair[0]+'+'+pair[1]
    np.savez_compressed(folder/(stem+'.npz'),T=T[sl],recovered=recovered,photographic_reference=ref_mask)
    with (folder/(stem+'.json')).open('w') as f:json.dump(result,f,indent=2)
    with (folder/(stem+'_curve.csv')).open('w',newline='') as f:
        wr=csv.DictWriter(f,fieldnames=['iteration','proxy_iou']);wr.writeheader();wr.writerows(curve)
    if plot:
        fig,axs=plt.subplots(1,4,figsize=(15,4))
        for ax,arr,title in zip(axs,[ref_mask,T[sl],recovered,ref_mask.astype(np.int8)-recovered.astype(np.int8)],
                   ['Photographic reference','Reconstructed T','Binary T > 0.50','Reference - reconstruction']):
            ax.imshow(arr,cmap='bwr' if 'Reference -' in title else 'gray');ax.set_title(title);ax.axis('off')
        fig.tight_layout()
        fig.savefig(folder/(stem+'.png'),dpi=120)
        plt.close(fig)
    print('Saved',folder/stem,'IoU',result['photographic_proxy_IoU'],flush=True)
    return result

## E. Run one plane pair (or all combinations)

**Native-grid runs are expensive.** The default performs two models for the chosen two-plane pair. Set `RUN_ALL_PAIRS=True` for 3 seven-hole or 15 three-hole pairs. This cell does not silently substitute a smaller detector grid.

In [ ]:
if RUN_ALL_PAIRS:
    ordered=sorted(Z,key=lambda n:Z[n])
    pairs=list(itertools.combinations(ordered,2))
else:
    pairs=[SELECTED_PAIR]
    if any(name not in Z for name in SELECTED_PAIR):
        raise ValueError('SELECTED_PAIR is not in the selected DATASET: '+str(tuple(Z)))
rows=[]
for method in METHODS:
    for pair in pairs:
        rows.append(reconstruct(pair,method,ITERATIONS))
with (OUT/DATASET/'all_scores.csv').open('w',newline='') as f:
    wr=csv.DictWriter(f,fieldnames=rows[0].keys());wr.writeheader();wr.writerows(rows)
print('Completed',len(rows),'native-resolution reconstructions')

## F. Sampling and accuracy checks

The original detector has approximately 2.775 μm pitch. Zero padding to 3072×2048 changes the FFT grid dimensions, not the physical sample pitch or measured information.

In [ ]:
print('Original image resolution',W,'x',H)
print('FFT padded resolution',PW,'x',PH)
print('Pixel pitch (micrometers)',PITCH*1e6)
print('Nyquist cycles per mm',1/(2*PITCH)/1000)
print('Completed scores:',OUT/DATASET/'all_scores.csv')